## Dataset Class

The `Dataset` class is used to define and organize your data. It acts as a blueprint for loading and accessing individual data samples.

When creating a custom dataset, you control:

* How data is loaded
* How many samples exist
* How each sample is returned

### Key Methods

* `__init__(self)`

  * Initializes the dataset
  * Used to load data (file paths, labels, etc.)

* `__len__(self)`

  * Returns the total number of samples in the dataset

* `__getitem__(self, index)`

  * Returns a single data sample (input and label) at a given index

### Example

```python
from torch.utils.data import Dataset

class MyDataset(Dataset):

    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        return self.data[index]
```

In [1]:
from sklearn.datasets import make_classification
import torch

In [2]:
# Create dummy data

# Step 1: Create a synthetic classification dataset using sklearn
X, y = make_classification(
    n_samples=10,       # Number of samples
    n_features=2,       # Number of features
    n_informative=2,    # Number of informative features
    n_redundant=0,      # Number of redundant features
    n_classes=2,        # Number of classes
    random_state=42     # For reproducibility
)

In [4]:
X , X.shape

(array([[ 1.06833894, -0.97007347],
        [-1.14021544, -0.83879234],
        [-2.8953973 ,  1.97686236],
        [-0.72063436, -0.96059253],
        [-1.96287438, -0.99225135],
        [-0.9382051 , -0.54304815],
        [ 1.72725924, -1.18582677],
        [ 1.77736657,  1.51157598],
        [ 1.89969252,  0.83444483],
        [-0.58723065, -1.97171753]]),
 (10, 2))

In [5]:
y

array([1, 0, 0, 0, 0, 1, 1, 1, 1, 0])

In [6]:
# Convert the data to PyTorch tensors
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)

In [7]:
from torch.utils.data import Dataset

In [16]:
class CustomDataset(Dataset):
    def __init__(self, features , labels):
        self.features = features
        self.labels = labels
        
    def __len__(self):
        return self.features.shape[0]

    def __getitem__(self, index):
        return self.features[index] , self.labels[index]

In [17]:
dataset = CustomDataset(X,y)

In [18]:
len(dataset)

10

In [20]:
dataset[0]

(tensor([ 1.0683, -0.9701]), tensor(1))

## DataLoader Class

The `DataLoader` class is used to efficiently load data from a dataset during training or testing. It automates batching, shuffling, and parallel data loading.

### Responsibilities

* Divides data into batches
* Shuffles data (if required)
* Loads data efficiently using multiple workers
* Iterates over the dataset

### Working Flow

1. At the start of each epoch, data is shuffled if `shuffle=True`
2. Indices are divided into batches based on `batch_size`
3. For each batch, data is fetched from the Dataset
4. Batches are returned one by one during iteration

### Example

```python
from torch.utils.data import DataLoader

data = [1, 2, 3, 4, 5, 6]

dataset = MyDataset(data)

loader = DataLoader(dataset, batch_size=2, shuffle=True)

for batch in loader:
    print(batch)
```

---

## Difference Between Dataset and DataLoader

| Feature | Dataset                | DataLoader             |
| ------- | ---------------------- | ---------------------- |
| Purpose | Defines data structure | Loads data efficiently |
| Output  | Single sample          | Batch of samples       |
| Control | Manual logic           | Automated handling     |
| Usage   | Data access            | Training loop input    |

In [21]:
from torch.utils.data import DataLoader

In [24]:
dataloader = DataLoader(dataset,batch_size=2,shuffle=False)

In [25]:
for batch_features , batch_labels in dataloader:
    print(batch_features)
    print(batch_labels)
    print("-"*100)

tensor([[ 1.0683, -0.9701],
        [-1.1402, -0.8388]])
tensor([1, 0])
----------------------------------------------------------------------------------------------------
tensor([[-2.8954,  1.9769],
        [-0.7206, -0.9606]])
tensor([0, 0])
----------------------------------------------------------------------------------------------------
tensor([[-1.9629, -0.9923],
        [-0.9382, -0.5430]])
tensor([0, 1])
----------------------------------------------------------------------------------------------------
tensor([[ 1.7273, -1.1858],
        [ 1.7774,  1.5116]])
tensor([1, 1])
----------------------------------------------------------------------------------------------------
tensor([[ 1.8997,  0.8344],
        [-0.5872, -1.9717]])
tensor([1, 0])
----------------------------------------------------------------------------------------------------


## DataLoader Parameters in PyTorch

The `DataLoader` class provides several parameters to control how data is loaded, batched, and processed during training.


### 1. `dataset` (mandatory)

* The dataset from which the DataLoader loads data
* Must be a subclass of `torch.utils.data.Dataset`
* Should implement:

  * `__getitem__()`
  * `__len__()`


### 2. `batch_size`

* Defines how many samples are included in one batch
* Default value is `1`
* Larger batch sizes:

  * Speed up training (especially on GPUs)
  * Require more memory



### 3. `shuffle`

* If `True`, data is shuffled at the start of each epoch
* Helps prevent the model from learning patterns based on data order
* Recommended for training datasets



### 4. `num_workers`

* Number of worker processes used for loading data
* `0` means data is loaded in the main process
* `> 0` enables parallel data loading

Advantages:

* Faster data loading
* Better CPU utilization
* Useful when preprocessing or disk I/O is slow



### 5. `pin_memory`

* If `True`, data is copied into pinned (page-locked) memory before being returned
* Improves data transfer speed from CPU to GPU
* Especially useful when using CUDA-enabled GPUs



### 6. `drop_last`

* If `True`, drops the last batch if it is smaller than `batch_size`
* Useful when equal batch sizes are required

Example:

* Dataset size = 10, batch_size = 4

  * `drop_last=False` → batches: [4, 4, 2]
  * `drop_last=True` → batches: [4, 4]



### 7. `collate_fn`

* A function used to combine multiple samples into a batch
* Default behavior:

  * Stacks tensors automatically

Custom use cases:

* Handling variable-length data (e.g., text sequences)
* Custom batching logic
* Complex data structures



### 8. `sampler`

* Defines how samples are selected from the dataset
* Useful for:

  * Imbalanced datasets
  * Custom sampling strategies

Related:

* `batch_sampler`:

  * Controls how batches are formed instead of individual samples

Note:

* Usually not needed if using `batch_size` and `shuffle`
* Used for advanced control over data loading



## Example

```python id="4zzdmu"
from torch.utils.data import DataLoader

loader = DataLoader(
    dataset=dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True
)
```



## Summary

* `dataset`: Source of data
* `batch_size`: Samples per batch
* `shuffle`: Randomizes data
* `num_workers`: Parallel loading
* `pin_memory`: Faster GPU transfer
* `drop_last`: Removes incomplete batch
* `collate_fn`: Custom batch creation
* `sampler`: Custom sampling strategy